In [5]:
import os
import pickle
import torch
import itertools
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from abc import ABC, abstractmethod
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def log_dataset_state(df, etapa):
    n_registros = len(df)
    n_publicacoes = df['work_id'].nunique() if 'work_id' in df.columns else 0
    n_autores = df['author_id'].nunique() if 'author_id' in df.columns else 0
    print(f"[{etapa}] registros={n_registros:,} | publicacoes={n_publicacoes:,} | autores_unicos={n_autores:,}")


def resolve_database_path(folder_name='database_50k'):
    # Suporta execucao com cwd em /link-prediction ou /link-prediction/marcos
    candidates = [Path.cwd() / folder_name, Path.cwd().parent / folder_name]
    for candidate in candidates:
        if (candidate / 'authorships.csv').exists() and (candidate / 'works.csv').exists():
            return str(candidate)
    raise FileNotFoundError(f'Nao foi possivel localizar a pasta {folder_name}. CWD atual: {Path.cwd()}')


database_path = resolve_database_path('database_50k')
authors_df = pd.read_csv(f'{database_path}/authorships.csv')
works_df = pd.read_csv(f'{database_path}/works.csv')

print('=== CARREGAMENTO INICIAL ===')
print(f"database_path resolvido: {database_path}")
print(f"authorships: {len(authors_df):,} linhas")
print(f"works: {len(works_df):,} linhas")

print('\n=== INTEGRACAO DAS TABELAS ===')
merged_df = authors_df.merge(
    works_df[['id', 'publication_date', 'title', 'abstract', 'language']],
    left_on='work_id', right_on='id'
    )
initial_records = len(merged_df)
log_dataset_state(merged_df, 'apos merge por work_id')

print('\n=== FILTRAGEM E PADRONIZACAO ===')
merged_df['publication_date'] = pd.to_datetime(merged_df['publication_date'], errors='coerce')
print('1) Datas padronizadas para datetime com coercao de erros.')

before_dropna = len(merged_df)
merged_df = merged_df.dropna(
    subset=['publication_date', 'author_id', 'title', 'abstract', 'language']
    ).drop(columns=['id'])
removed_dropna = before_dropna - len(merged_df)
pct_dropna = (removed_dropna / before_dropna * 100) if before_dropna else 0
print(f"2) Remocao de nulos em campos essenciais: -{removed_dropna:,} registros ({pct_dropna:.2f}%)")
log_dataset_state(merged_df, 'apos remocao de nulos')

before_lang = len(merged_df)
merged_df = merged_df[merged_df['language'] == 'en']
removed_lang = before_lang - len(merged_df)
pct_lang = (removed_lang / before_lang * 100) if before_lang else 0
print(f"3) Filtro de idioma (en): -{removed_lang:,} registros ({pct_lang:.2f}%)")
log_dataset_state(merged_df, 'apos filtro idioma en')

final_records = len(merged_df)
total_removed = initial_records - final_records
pct_total_removed = (total_removed / initial_records * 100) if initial_records else 0
pct_retained = (final_records / initial_records * 100) if initial_records else 0
print(f"4) Variacao total: removidos={total_removed:,} ({pct_total_removed:.2f}%) | mantidos={final_records:,} ({pct_retained:.2f}%)")

min_date = merged_df['publication_date'].min()
max_date = merged_df['publication_date'].max()
min_year = int(min_date.year) if pd.notna(min_date) else None
max_year = int(max_date.year) if pd.notna(max_date) else None

unique_works = merged_df[['work_id', 'publication_date']].drop_duplicates().sort_values('publication_date')
split_idx = int(len(unique_works) * 0.8)

train_work_ids = set(unique_works.iloc[:split_idx]['work_id'])
test_work_ids = set(unique_works.iloc[split_idx:]['work_id'])

train_df = merged_df[merged_df['work_id'].isin(train_work_ids)]
test_df = merged_df[merged_df['work_id'].isin(test_work_ids)]
print('\n=== SPLIT TEMPORAL 80/20 POR PUBLICACAO ===')
print(f"train works: {len(train_work_ids):,} | test works: {len(test_work_ids):,}")


def build_graph(df):
    graph = defaultdict(set)
    for _, group in df.groupby('work_id'):
        authors = group['author_id'].tolist()

        if len(authors) > 1:
            for u, v in itertools.permutations(authors, 2):
                graph[u].add(v)

    return graph


train_graph = build_graph(train_df)
test_graph_raw = build_graph(test_df)

# Media de coautores por autor no treino (considerando autores com pelo menos 1 coautor)
train_coauthor_counts = [len(coauthors) for coauthors in train_graph.values()]
avg_coauthors_train = float(np.mean(train_coauthor_counts)) if train_coauthor_counts else 0.0

test_ground_truth = defaultdict(set)

for author, coauthors in test_graph_raw.items():
    # Coautorias futuras no conjunto de teste
    future_coauthors = coauthors

    # Remove coautorias ja observadas no treino (foco em novos links)
    past_coauthors = train_graph.get(author, set())
    new_links = future_coauthors - past_coauthors

    if new_links:
        test_ground_truth[author] = new_links

print('\n=== RESUMO FINAL DO DATASET ===')
dataset_summary = pd.DataFrame([{
    'registros_autoria': len(merged_df),
    'publicacoes_distintas': merged_df['work_id'].nunique(),
    'autores_unicos': merged_df['author_id'].nunique(),
    'idioma': 'Ingles',
    'periodo_publicacao': f'{min_year}--{max_year}',
    'media_coautores_treino': round(avg_coauthors_train, 2)
}])
print(dataset_summary.to_string(index=False))

print('\n=== TEXTO-BASE (VALORES ATUALIZADOS) ===')
print(
    f"Dataset final: {len(merged_df):,} registros de autoria, "
    f"{merged_df['work_id'].nunique():,} publicacoes distintas, "
    f"{merged_df['author_id'].nunique():,} autores unicos, "
    f"idioma Ingles, periodo {min_year}-{max_year}, "
    f"media de coautores por autor (treino): {avg_coauthors_train:.2f}."
)

print('\n=== GROUND TRUTH DE NOVOS LINKS ===')
print(f"autores com novos links no teste: {len(test_ground_truth):,}")

C:\Users\BSBCo\AppData\Local\Temp\ipykernel_25256\4130470832.py:34: DtypeWarning: Columns (0: is_retracted) have mixed types. Specify dtype option on import or set low_memory=False.
  works_df = pd.read_csv(f'{database_path}/works.csv')


=== CARREGAMENTO INICIAL ===
database_path resolvido: c:\Users\BSBCo\dev\link-prediction\database_50k
authorships: 189,283 linhas
works: 40,302 linhas

=== INTEGRACAO DAS TABELAS ===
[apos merge por work_id] registros=189,283 | publicacoes=40,302 | autores_unicos=50,146

=== FILTRAGEM E PADRONIZACAO ===
1) Datas padronizadas para datetime com coercao de erros.
2) Remocao de nulos em campos essenciais: -74,571 registros (39.40%)
[apos remocao de nulos] registros=114,712 | publicacoes=25,768 | autores_unicos=35,253
3) Filtro de idioma (en): -3,587 registros (3.13%)
[apos filtro idioma en] registros=111,125 | publicacoes=24,896 | autores_unicos=33,800
4) Variacao total: removidos=78,158 (41.29%) | mantidos=111,125 (58.71%)

=== SPLIT TEMPORAL 80/20 POR PUBLICACAO ===
train works: 19,916 | test works: 4,980

=== RESUMO FINAL DO DATASET ===
 registros_autoria  publicacoes_distintas  autores_unicos idioma periodo_publicacao  media_coautores_treino
            111125                  24896   

In [15]:

print('\n=== INFORMACOES PARA ATUALIZACAO DO TEXTO ===')
print(f"Registros de autoria (após filtragem): {len(merged_df):,}")
print(f"Publicações distintas: {merged_df['work_id'].nunique():,}")
print(f"Publicações no treino: {len(train_work_ids):,}")
print(f"Publicações no teste: {len(test_work_ids):,}")
print(f"Período temporal: {min_year}--{max_year}")
print(f"Autores no grafo de treino: {len(train_graph):,}")
print(f"Média de coautores por autor (treino): {avg_coauthors_train:.2f}")
print(f"Autores no grafo de teste: {len(test_graph_raw):,}")
print(f"Autores com novas conexões: {len(test_ground_truth):,}")
authors_no_new_links = len(test_graph_raw) - len(test_ground_truth)
pct_new_links = (len(test_ground_truth) / len(test_graph_raw) * 100) if len(test_graph_raw) > 0 else 0
pct_no_new_links = (authors_no_new_links / len(test_graph_raw) * 100) if len(test_graph_raw) > 0 else 0
print(f"Autores sem novas conexões: {authors_no_new_links:,} ({pct_no_new_links:.1f}%)")
print(f"Autores com novas conexões (percentual): {len(test_ground_truth):,} ({pct_new_links:.1f}%)")

# Média de coautores por autor no teste
test_coauthor_counts = [len(coauthors) for coauthors in test_graph_raw.values()]
avg_coauthors_test = float(np.mean(test_coauthor_counts)) if test_coauthor_counts else 0.0
print(f"Média de coautores por autor (teste): {avg_coauthors_test:.2f}")



=== INFORMACOES PARA ATUALIZACAO DO TEXTO ===
Registros de autoria (após filtragem): 101,692
Publicações distintas: 22,302
Publicações no treino: 19,916
Publicações no teste: 4,980
Período temporal: 1792--2026
Autores no grafo de treino: 25,260
Média de coautores por autor (treino): 17.22
Autores no grafo de teste: 11,441
Autores com novas conexões: 11,066
Autores sem novas conexões: 375 (3.3%)
Autores com novas conexões (percentual): 11,066 (96.7%)
Média de coautores por autor (teste): 17.15
